# Erzeugung der Embeddings

In diesem Notebook werden die im Rahmen der systematischen Literaturrecherche (SLR) identifizierten Benchmark-Datensätze geladen und für die Vektorisierung vorbereitet. Dies umfasst sowohl den generischen Referenz-Datensatz als auch die spezialisierten Benchmarks aus den Domänen Medizin, Recht und Finanzwesen.

## Vorbereitung

Zunächst werden die benötigten Bibliotheken importiert.

In [18]:
import numpy as np
from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pandas as pd
from IPython.display import display, HTML
import warnings
import json

Anschließend werden die Dateipfade für den Datenzugriff definiert.

In [19]:
# Basis
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# generisch 
GENERIC_CORPUS_DIR  = DATA_DIR / "generic" / "corpus"
GENERIC_QUERIES_DIR = DATA_DIR / "generic" / "queries"  
GENERIC_IDS_DIR     = DATA_DIR / "generic" / "ids"       
GENERIC_SPLIT_DIR   = Path("/mnt/user/experiment/data/generic/splits")

# medizinisch
MED_CORPUS_DIR = DATA_DIR / "medicine" / "corpus"
MED_QUERIES_DIR = DATA_DIR / "medicine" / "queries"
MED_IDS_DIR = DATA_DIR / "medicine" / "ids"

# recht - LegalBenchRAG Rohdaten 
LAW_RAW_DIR       = DATA_DIR / "law" / "legalbenchrag"
LAW_RAW_CORPUS    = LAW_RAW_DIR / "corpus"
LAW_RAW_BENCHMARKS= LAW_RAW_DIR / "benchmarks"
# recht - nach Bearbeitung der Rohdaten 
LAW_CORPUS_DIR = DATA_DIR / "law" / "corpus"
LAW_QUERIES_DIR = DATA_DIR / "law" / "queries"
LAW_IDS_DIR = DATA_DIR / "law" / "ids"

# finanzen
FIN_CORPUS_DIR = DATA_DIR / "finance" / "corpus"
FIN_QUERIES_DIR = DATA_DIR / "finance" / "queries"
FIN_IDS_DIR = DATA_DIR / "finance" / "ids"

Basierend auf der SLR wurde festgestellt, dass das Modell **all-MiniLM-L6-v2** in der einschlägigen Literatur am häufigsten verwendet wird. Dementsprechend wird dieses Modell über die `sentence-transformers`-Bibliothek geladen.

In [20]:
# Modell laden
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Benchmarks 

Für die Erzeugung der Embeddings werden basierend auf der SLR ausgewählte Benchmarks verwendet.

### Generischer Benchmark: Natural Questions (NQ)

In dieser Arbeit wird der **Natural Questions (NQ)** als generischer Datensatz aus dem BEIR-Benchmark verwendet.
Um die Recheneffizienz zu wahren und dennoch eine statistisch signifikante Varianz abzubilden, wird ein Split von **100.000 Dokumenten** geladen.

In [7]:
print("\nLade NQ-Datensatz (100.000 Texte)...")

# 1. Corpus laden (auf 100.000 begrenzt)
nq_corpus = load_dataset("beir/nq", "corpus", split="corpus[:100000]")
print(f"Corpus: {nq_corpus.num_rows:,} Dokumente")
print(f"Spalten: {nq_corpus.column_names}\n")

# Set der gültigen Dokument-IDs
gueltige_doc_ids = set(nq_corpus["_id"])

# 2. Alle Qrels laden und auf die 100.000 Docs filtern
nq_qrels = load_dataset("beir/nq-qrels", split="test")
qrels_df = nq_qrels.to_pandas()
qrels_gefiltert = qrels_df[qrels_df["corpus-id"].isin(gueltige_doc_ids)]
gueltige_query_ids = set(qrels_gefiltert["query-id"])
print(f"Qrels: {len(qrels_gefiltert)} Zuordnungen (gefiltert auf die 100k Docs)")

# 3. Alle Queries laden und auf die passenden filtern
nq_queries = load_dataset("beir/nq", "queries", split="queries")
queries_df = nq_queries.to_pandas()
queries_gefiltert = queries_df[queries_df["_id"].isin(gueltige_query_ids)]
print(f"Queries: {len(queries_gefiltert)} Fragen (die zu den 100k Docs passen)\n")

# Vorschau
pd.set_option('display.max_colwidth', None)
print("--- Corpus (3 Beispiele) ---")
display(nq_corpus.select(range(3)).to_pandas())
print("--- Queries (3 Beispiele) ---")
display(queries_gefiltert.head(3))
print("--- Qrels (3 Beispiele) ---")
display(qrels_gefiltert.head(3))


Lade NQ-Datensatz (100.000 Texte)...
Corpus: 100,000 Dokumente
Spalten: ['_id', 'title', 'text']

Qrels: 3554 Zuordnungen (gefiltert auf die 100k Docs)
Queries: 2924 Fragen (die zu den 100k Docs passen)

--- Corpus (3 Beispiele) ---


,_id,title,text
0,doc0,Minority interest,"In accounting, minority interest (or non-controlling interest) is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]"
1,doc1,Minority interest,"It is, however, possible (such as through special voting rights) for a controlling interest requiring consolidation to be achieved without exceeding 50% ownership, depending on the accounting standards being employed. Minority interest belongs to other investors and is reported on the consolidated balance sheet of the owning company to reflect the claim on assets belonging to other, non-controlling shareholders. Also, minority interest is reported on the consolidated income statement as a share of profit belonging to minority shareholders."
2,doc2,Minority interest,"The reporting of 'minority interest' is a consequence of the requirement by accounting standards to 'fully' consolidate partly owned subsidiaries. Full consolidation, as opposed to partial consolidation, results in financial statements that are constructed as if the parent corporation fully owns these partly owned subsidiaries; except for two line items that reflect partial ownership of subsidiaries: net income to common shareholders and common equity. The two minority interest line items are the net difference between what would have been the common equity and net income to common, if all subsidiaries were fully owned, and the actual ownership of the group. All the other line items in the financial statements assume a fictitious 100% ownership."


--- Queries (3 Beispiele) ---


,_id,title,text
0,test0,,what is non controlling interest on balance sheet
1,test1,,how many episodes are in chicago fire season 4
2,test2,,who sings love will keep us alive by the eagles


--- Qrels (3 Beispiele) ---


,query-id,corpus-id,score
0,test0,doc0,1
1,test0,doc1,1
2,test1,doc6,1


Für die Vektorisierung werden der Titel und der Haupttext jedes Dokuments konkateniert, um den vollen semantischen Kontext zu erfassen. Die Erzeugung der Embeddings erfolgt unter Nutzung der GPU-Beschleunigung (CUDA), wobei die Roh-Vektoren anschließend für die spätere Verwendung in der PCA-Pipeline persistiert werden. Die Qrels werden als JSON gespeichert, da eine Query mehreren relevanten Dokumenten zugeordnet sein kann und dieses Format die 1:n-Beziehung abbildet.

In [27]:
# Texte + IDs extrahieren
print("Extrahiere Corpus-Texte...")
corpus_texts = []
corpus_ids = []
for item in nq_corpus:
    if item["title"]:
        corpus_texts.append(f"{item['title']} : {item['text']}")
    else:
        corpus_texts.append(item["text"])
    corpus_ids.append(item["_id"]) 

print("Extrahiere Query-Texte...")
query_texts = queries_gefiltert["text"].tolist()
query_ids   = queries_gefiltert["_id"].tolist()  

# Qrels als Lookup speichern 
qrels_dict = {}
for _, row in qrels_gefiltert.iterrows():
    q_id = row["query-id"]
    c_id = row["corpus-id"]
    if q_id not in qrels_dict:
        qrels_dict[q_id] = []
    qrels_dict[q_id].append(c_id)

print(f"{len(corpus_texts)} Corpus-Texte, {len(query_texts)} Fragen bereit.\n")

# Embeddings berechnen
print("Berechne Corpus Embeddings...")
corpus_vecs = model.encode(corpus_texts, show_progress_bar=True, batch_size=1024, device="cuda")

print("Berechne Query Embeddings...")
query_vecs = model.encode(query_texts, show_progress_bar=True, batch_size=1024, device="cuda")

# Speichern
print("\nSpeichere NQ-Matrizen...")
np.save(GENERIC_CORPUS_DIR  / "generic_corpus_vecs.npy", corpus_vecs)
np.save(GENERIC_QUERIES_DIR / "generic_query_vecs.npy",  query_vecs)
np.save(GENERIC_IDS_DIR     / "generic_corpus_ids.npy",  np.array(corpus_ids))
np.save(GENERIC_IDS_DIR     / "generic_query_ids.npy",   np.array(query_ids))

# Qrels als JSON (weil 1 Query -> mehrere Docs möglich)
with open(GENERIC_IDS_DIR / "generic_qrels.json", "w") as f:
    json.dump(qrels_dict, f)

print("NQ erfolgreich verarbeitet und gespeichert!")

Extrahiere Corpus-Texte...
Extrahiere Query-Texte...
100000 Corpus-Texte, 2924 Fragen bereit.

Berechne Corpus Embeddings...


Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Berechne Query Embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


Speichere NQ-Matrizen...
NQ erfolgreich verarbeitet und gespeichert!


### Domänenspezifischer Benchmark: Medizin (PubMedQA)

Um die Leistungsfähigkeit der domänenspezifischen PCA im Vergleich zur generischen Variante zu untersuchen, wird als erster Fachbereich die **Medizin** herangezogen. Hierfür wird der **PubMedQA**-Datensatz verwendet. <br>
Dieser Benchmark basiert auf biomedizinischen Abstracts aus der PubMed-Datenbank. 

In [6]:
print("\nLade PubMedQA Datensatz...")
med_dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(f"Gesamtgröße des geladenen Splits: {med_dataset.num_rows:,} Frage-Antwort-Paare")
print(f"Enthaltene Spalten: {med_dataset.column_names}\n")
pd.set_option('display.max_colwidth', None) 
display(med_dataset.select(range(3)).to_pandas())


Lade PubMedQA Datensatz...
Gesamtgröße des geladenen Splits: 1,000 Frage-Antwort-Paare
Enthaltene Spalten: ['pubid', 'question', 'context', 'long_answer', 'final_decision']



,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?,"{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos and examined. Mitochondrial dynamics were delineated into four categories (M1-M4) based on characteristics including distribution, motility, and membrane potential (ΔΨm). A TUNEL assay showed fragmented nDNA in a gradient over these mitochondrial stages. Chloroplasts and transvacuolar strands were also examined using live cell imaging. The possible importance of mitochondrial permeability transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves with a significantly lower number of perforations compared to controls, and that displayed mitochondrial dynamics similar to that of non-PCD cells.'], 'labels': ['BACKGROUND', 'RESULTS'], 'meshes': ['Alismataceae', 'Apoptosis', 'Cell Differentiation', 'Mitochondria', 'Plant Leaves'], 'reasoning_required_pred': ['y', 'e', 's'], 'reasoning_free_pred': ['y', 'e', 's']}","Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.",yes
1,16418930,Landolt C and snellen e acuity: differences in strabismus amblyopia?,"{'contexts': ['Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if their critical details appear under the same visual angle. Since optotypes are evaluated on individuals with good visual acuity and without eye disorders, differences in the lower visual acuity range cannot be excluded. In this study, visual acuity measured with the Snellen E was compared to the Landolt C acuity.', '100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with amblyopia due to strabismus, and 13 healthy volunteers were tested. Charts with the Snellen E and the Landolt C (Precision Vision) which mimic the ETDRS charts were used to assess visual acuity. Three out of 5 optotypes per line had to be correctly identified, while wrong answers were monitored. In the group of patients, the eyes with the lower visual acuity, and the right eyes of the healthy subjects, were evaluated.', 'Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal values for LR and SE were 0.25 and 0.29 in the entire group and 0.14 and 0.16 for the eyes with strabismus amblyopia. 

Nach dem Laden des PubMedQA-Datensatzes werden drei zentrale Komponenten extrahiert:

* **Corpus (`context`):** Die verschachtelte Dictionary-Struktur wird aufgelöst und die medizinischen Sätze zu flachen Text-Strings zusammengefügt. Diese bilden den Suchraum in der FAISS-Datenbank.
* **Queries (`question`):** Die medizinischen Fragen dienen als Suchbefehle für den Retriever. Auf die Langantworten (`long_answer`) wird bewusst verzichtet, da nur die Retrieval-Qualität evaluiert wird.
* **Pubid (Ground Truth):** Jeder Corpus- und Query-Vektor wird mit derselben `pubid` verknüpft. Bei der Evaluation wird geprüft, ob die Top-Ergebnisse dieselbe ID wie die Frage besitzen. 

In [8]:
corpus_texts = []
corpus_ids = []
query_texts = []
query_ids = []

print("\nExtrahiere Dokumente und Fragen...")
for item in med_dataset:
    # ID
    doc_id = str(item["pubid"])
    
    # Corpus
    raw_context = item["context"]
    # Wenn Dictionary, hol die Sätze raus. Sonst mach es zu einem String.
    if isinstance(raw_context, dict):
        doc_text = " ".join(raw_context.get("contexts", []))
    else:
        doc_text = str(raw_context)       
    corpus_texts.append(doc_text)
    corpus_ids.append(doc_id)
    
    # Queries
    query_text = str(item["question"])   
    query_texts.append(query_text)
    query_ids.append(doc_id) # WICHTIG: Die Frage bekommt exakt dieselbe ID wie das Dokument!

print(f"{len(corpus_texts)} Dokumente und {len(query_texts)} Fragen sauber extrahiert.")

print("\nBerechne Corpus Embeddings (Dokumente)...")
corpus_vecs = model.encode(
    corpus_texts, 
    show_progress_bar=True, 
    batch_size=1024, 
    device="cuda"
)

print("Berechne Query Embeddings (Fragen)...")
query_vecs = model.encode(
    query_texts, 
    show_progress_bar=True, 
    batch_size=1024, 
    device="cuda"
)
print("\nSpeichere Medizin-Matrizen auf die Festplatte...")
# Corpus
np.save(MED_CORPUS_DIR / "medicine_corpus_vecs.npy", corpus_vecs)
# Queries 
np.save(MED_QUERIES_DIR / "medicine_query_vecs.npy", query_vecs)
# IDs
np.save(MED_IDS_DIR / "medicine_corpus_ids.npy", np.array(corpus_ids))
np.save(MED_IDS_DIR / "medicine_query_ids.npy", np.array(query_ids))

print("Medizin-Domäne verarbeitet und gespeichert!")


Extrahiere Dokumente und Fragen...
1000 Dokumente und 1000 Fragen sauber extrahiert.

Berechne Corpus Embeddings (Dokumente)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Berechne Query Embeddings (Fragen)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Speichere Medizin-Matrizen auf die Festplatte...
Medizin-Domäne verarbeitet und gespeichert!


### Domänenspezifischer Benchmark: Recht (Legal-BenchRAG)

Als zweiter Fachbereich wird das Recht untersucht. Hierfür wird der **LegalBench-RAG**-Datensatz verwendet, ein Benchmark für juristische Retrieval-Aufgaben. Die Daten wurden manuell aus dem zugehörigen GitHub-Repository heruntergeladen und liegen in entsprechenden Dateien vor.

In [8]:
benchmark_files = sorted(LAW_RAW_BENCHMARKS.glob("*.json"))
print(f"Gefundene Benchmark-Dateien: {[f.name for f in benchmark_files]}\n")

with open(benchmark_files[0]) as f:
    bm = json.load(f)

test_cases = bm["tests"]  
print(f"Benchmark: {benchmark_files[0].name}")
print(f"Anzahl Testfälle: {len(test_cases)}\n")

preview_rows = []
for tc in test_cases[:3]:
    preview_rows.append({
        "query": tc["query"],
        "gt_file": tc["snippets"][0]["file_path"],  
        "char_start": tc["snippets"][0]["span"][0],  
        "char_end":   tc["snippets"][0]["span"][1],  
    })

pd.set_option("display.max_colwidth", None)
display(HTML("<b>Benchmark JSON – erste 3 Einträge:</b>"))
display(pd.DataFrame(preview_rows))

Gefundene Benchmark-Dateien: ['contractnli.json', 'cuad.json', 'maud.json', 'privacy_qa.json']

Benchmark: contractnli.json
Anzahl Testfälle: 977



,query,gt_file,char_start,char_end
0,Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?,contractnli/CopAcc_NDA-and-ToP-Mentors_2.0_2017.txt,11461,11963
1,Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document state that Confidential Information shall only include technical information?,contractnli/CopAcc_NDA-and-ToP-Mentors_2.0_2017.txt,7752,8016
2,Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document mention that some obligations of the Agreement may survive the termination of the Agreement?,contractnli/CopAcc_NDA-and-ToP-Mentors_2.0_2017.txt,12076,12328


Aus dem Datensatz werden drei zentrale Komponenten extrahiert:

* **Corpus:** Die juristischen Dokumente werden als `.txt`-Dateien eingelesen, wobei der relative Dateipfad als eindeutige ID dient.
* **Queries:** Die Benchmark-Fragen werden aus den JSON-Dateien extrahiert.
* **`file_path` (Ground Truth):** Als *Ground Truth* wird der `file_path` des zugehörigen Snippets verwendet. Ein Treffer liegt vor, wenn das Retrieval dasselbe Dokument liefert.

In [21]:
corpus_texts = []
corpus_ids = []
query_texts = []
query_ids = []

# Corpus
print("Lese Corpus-Dateien ein...")
for txt_file in sorted(LAW_RAW_CORPUS.rglob("*.txt")):
    # extrahiert echt Pfade von den dokuemnten, die als ID dienen
    file_id = str(txt_file.relative_to(LAW_RAW_CORPUS))  
    text = txt_file.read_text(encoding="utf-8", errors="ignore")
    corpus_texts.append(text)
    corpus_ids.append(file_id)

print(f"{len(corpus_texts)} Textdateien als Corpus geladen.\n")

# Queries
print("Lese Benchmark-Fragen ein...")
for bm_file in sorted(LAW_RAW_BENCHMARKS.glob("*.json")):
    with open(bm_file) as f:
        data = json.load(f)
    for tc in data["tests"]:
        query_texts.append(tc["query"])
        gt_file_id = tc["snippets"][0]["file_path"]  
        query_ids.append(gt_file_id)

print(f"{len(query_texts)} Fragen geladen.\n")

# Embeddings
print("Berechne Corpus Embeddings (Verträge)...")
corpus_vecs = model.encode(corpus_texts, show_progress_bar=True, batch_size=1024, device="cuda")

print("Berechne Query Embeddings (Fragen)...")
query_vecs = model.encode(query_texts, show_progress_bar=True, batch_size=1024, device="cuda")

# Speichern
print("\nSpeichere Legal-Matrizen...")
np.save(LAW_CORPUS_DIR  / "law_corpus_vecs.npy", corpus_vecs)
np.save(LAW_QUERIES_DIR / "law_query_vecs.npy",  query_vecs)
np.save(LAW_IDS_DIR     / "law_corpus_ids.npy",  np.array(corpus_ids))
np.save(LAW_IDS_DIR     / "law_query_ids.npy",   np.array(query_ids))

print(f"\nCorpus Shape: {corpus_vecs.shape}")
print(f"Query Shape:  {query_vecs.shape}")
print("Recht-Domäne (LegalBenchRAG) verarbeitet und gespeichert!")

Lese Corpus-Dateien ein...
698 Textdateien als Corpus geladen.

Lese Benchmark-Fragen ein...
6889 Fragen geladen.

Berechne Corpus Embeddings (Verträge)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Berechne Query Embeddings (Fragen)...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Speichere Legal-Matrizen...

Corpus Shape: (698, 384)
Query Shape:  (6889, 384)
Recht-Domäne (LegalBenchRAG) verarbeitet und gespeichert!


### Domänenspezifischer Benchmark: Finanzen (FinanceBench)

Den Abschluss der Datenerhebung bildet die Finanzdomäne unter Verwendung des **FinanceBench**-Datensatzes. Dieser Benchmark basiert auf offiziellen Geschäftsberichten börsennotierter Unternehmen. 

In [5]:
print("Lade FinanceBench Datensatz...")
finance_ds = load_dataset("PatronusAI/financebench", split="train")
print(f"Gesamtgröße: {finance_ds.num_rows:,} Frage-Beweis-Paare")
print(f"Enthaltene Spalten: {finance_ds.column_names}\n")
pd.set_option('display.max_colwidth', None) 
display(finance_ds.select(range(3)).to_pandas())

Lade FinanceBench Datensatz...
Gesamtgröße: 150 Frage-Beweis-Paare
Enthaltene Spalten: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']



financebench_id company     doc_name      question_type  \
0  financebench_id_03029      3M  3M_2018_10K  metrics-generated   
1  financebench_id_04672      3M  3M_2018_10K  metrics-generated   
2  financebench_id_00499      3M  3M_2022_10K    domain-relevant   

                                 question_reasoning domain_question_num  \
0                            Information extraction                None   
1                            Information extraction                None   
2  Logical reasoning (based on numerical reasoning)                dg06   

                                                                                                                                                                                                               question  \
0                                                   What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.   
1  Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.   
2                                                                                                                                                              Is 3M a capital-intensive business based on FY2022 data?   

                                                                                                                                                                                                    answer  \
0                                                                                                                                                                                                 $1577.00   
1                                                                                                                                                                                                    $8.70   
2  No, the company is managing its CAPEX and Fixed Assets pretty efficiently, which is evident from below key metrics:\nCAPEX/Revenue Ratio: 5.1%\nFixed assets/Total Assets: 20%\nReturn on Assets= 12.4%   

                                                                                                                                                                  justification  \
0  The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).   
1                         The metric ppne, net was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Property, plant and equipment â net.   
2                                                                                                         CAPEX/Revenue\nFixed Assets/Total Assets\nROA=Net Income/Total Assets   

  dataset_subset_label  \
0          OPEN_SOURCE   
1          OPEN_SOURCE   
2          OPEN_SOURCE   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

Aus dem Datensatz werden folgende Komponenten extrahiert:

* **Corpus (`evidence`, `company`, `doc_name`):** Per Context-Injection werden Firmenname und Berichtsname jedem Textsegment vorangestellt (z. B. „Apple | 2023 Q3 Report : …"), um anonyme Passagen zu vermeiden und die Cosine Similarity bei unternehmensspezifischen Anfragen zu verbessern.
* **Queries (`question`):** Analytische Finanzfragen, wie sie von Analysten gestellt werden, dienen als Suchbefehle für den Retriever.
* **Ground Truth (`financebench_id`):** Jeder Query und dem zugehörigen Corpus-Eintrag wird dieselbe ID zugeordnet. Ein Treffer liegt vor, wenn das Retrieval-Ergebnis die korrekte ID liefert.

In [12]:
corpus_texts = []
corpus_ids = []
query_texts = []
query_ids = []

print("\nExtrahiere Finanzberichte (Corpus) und Fragen (Queries)...")
for item in finance_ds:
    # ID
    doc_id = str(item["financebench_id"])
    
    # Corpus
    company = str(item.get("company", "Unknown Company"))
    doc_name = str(item.get("doc_name", "Unknown Report"))
    
    raw_evidence = item["evidence"]
    extracted_text = ""
    
    # Entpackung
    if isinstance(raw_evidence, list):
        parts = []
        for e in raw_evidence:
            if isinstance(e, dict) and "evidence_text" in e:
                parts.append(str(e["evidence_text"]))
            else:
                parts.append(str(e))
        extracted_text = " ".join(parts)
    else:
        extracted_text = str(raw_evidence)
    
    # String für FAISS
    combined_corpus_text = f"{company} | {doc_name} : {extracted_text}"
    
    corpus_texts.append(combined_corpus_text)
    corpus_ids.append(doc_id)
    
    # Die Queries
    query_texts.append(str(item["question"]))
    query_ids.append(doc_id) # ID spiegeln

print(f"{len(corpus_texts)} Berichte und {len(query_texts)} Fragen sauber extrahiert.")

print("\nBerechne Corpus Embeddings (Berichte)...")
corpus_vecs = model.encode(
    corpus_texts, 
    show_progress_bar=True, 
    batch_size=1024, 
    device="cuda"
)

print("Berechne Query Embeddings (Fragen)...")
query_vecs = model.encode(
    query_texts, 
    show_progress_bar=True, 
    batch_size=1024, 
    device="cuda"
)

print("\nSpeichere Finanz-Matrizen auf die Festplatte...")
# Corpus
np.save(FIN_CORPUS_DIR / "finance_corpus_vecs.npy", corpus_vecs)
# Queries
np.save(FIN_QUERIES_DIR / "finance_query_vecs.npy", query_vecs)
# IDs
np.save(FIN_IDS_DIR / "finance_corpus_ids.npy", np.array(corpus_ids))
np.save(FIN_IDS_DIR / "finance_query_ids.npy", np.array(query_ids))

print("Finanzen-Domäne erfolgreich verarbeitet und gespeichert")


Extrahiere Finanzberichte (Corpus) und Fragen (Queries)...
150 Berichte und 150 Fragen sauber extrahiert.

Berechne Corpus Embeddings (Berichte)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Berechne Query Embeddings (Fragen)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Speichere Finanz-Matrizen auf die Festplatte...
Finanzen-Domäne erfolgreich verarbeitet und gespeichert
